#Para o Seminário de Sondagem

Este script realiza o download, processamento e análise de dados de sondagem atmosférica utilizando bibliotecas Python como o MetPy.

As principais etapas realizadas neste script são:

Instalação das bibliotecas necessárias: Instala o MetPy, Siphon e html2text.

Importação de bibliotecas: Importa as bibliotecas necessárias para a análise e geração de gráficos.

Configuração de parâmetros: Define o ID da estação, a data e a hora para os dados de sondagem.

Obtenção de dados: Realiza o download dos dados de sondagem do site Wyoming Upper Air Soundings utilizando o Siphon.

Cálculo de variáveis atmosféricas: Calcula diversos parâmetros atmosféricos como temperatura potencial, temperatura potencial equivalente, umidade relativa, razão de mistura, estabilidade estática e frequência de Brunt-Vaisala utilizando o MetPy.

Apresentação das variáveis calculadas: Apresenta as variáveis calculadas em formato de tabela utilizando um DataFrame do pandas.

Plotagem de variáveis atmosféricas: Gera gráficos das temperaturas potenciais, da frequência de Brunt-Vaisala e da estabilidade estática em função da pressão.

Download e processamento de dados de sondagem: Realiza o download dos dados de sondagem em um formato diferente e os processa para uso na sondagem

Execução de código Fortran: Executa um programa em Fortran (wyoming.f) para realizar cálculos adicionais (provavelmente relacionados a diagramas termodinâmicos).

Plotagem de diagramas termodinâmicos: Gera gráficos de contorno das diferenças de temperatura de densidade utilizando a saída do programa em Fortran.

> **Versão revisada (out/2026).** Mesma estrutura do notebook original, com estas mudanças:
> * a busca dos dados usa a interface atual do Wyoming (`wsgi`) e **nunca** troca por dados de exemplo — se não houver dados, a célula para com erro;
> * o `sounding.txt` lido pelo Fortran é escrito **a partir da mesma sondagem** analisada (antes a URL apontava para outra estação e outra data);
> * os índices calculados pelo próprio Wyoming (`type=INDICES`) são mostrados como **guia de conferência**;
> * correções: Lifted Index (recebia Td no lugar do perfil da parcela), SRH relativa ao movimento de Bunkers (HS: *left-mover*, SRH ciclônica negativa),
>   alturas observadas (não atmosfera padrão), unidade da estabilidade estática, leitura dos arquivos `.out`, `savefig` antes de `show`;
> * removidos: segunda cópia do Skew-T, índices com definição errada (SigSevere, "effective" SRH/shear, MESH inventado) e a análise por IA;
> * **nova seção final:** as três sondagens (estável, neutra, instável) — Skew-T, θ/θe/θes e matrizes de Emanuel.
>
> **Regra:** todo número usado na apresentação deve sair das contas deste notebook.

In [ ]:
!pip install -q metpy siphon
!which gfortran > /dev/null || sudo apt-get -qq install gfortran

In [ ]:
from matplotlib import transforms
%matplotlib inline

In [ ]:
from datetime import datetime
from metpy.units import units
from siphon.simplewebservice.wyoming import WyomingUpperAir

%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import pandas as pd
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import Hodograph, SkewT
from metpy.units import units
import numpy as np
import subprocess
import os, re, requests

In [ ]:
df=[]

In [ ]:
station = '83971' # Porto Alegre (SBPA). Exemplo para Floripa é 83899
yy=1995
mm=12
dd=24
hh=12
date = datetime(yy, mm, dd, hh) # Cuidado com os horarios (normalmente tem as 12Z)
src = 'FM35'   # 'FM35' para dados antigos (ex.: 1995) | 'BUFR' para dados recentes
f = open('header.txt', "w+")
f.write(station+' '+str(yy)+' '+str(mm)+' '+str(dd)+' '+str(hh))
f.write('\n')
f.close()
!cat header.txt

In [ ]:
# ====================================================================
# Busca da sondagem na Universidade de Wyoming (interface wsgi)
#  - tabela da sondagem: type=TEXT:LIST
#  - índices calculados pelo Wyoming: type=INDICES (guia de conferência)
#  - escreve sounding.txt no formato que o wyoming.f (K. Emanuel) lê
# NÃO existe dado de exemplo substituto: sem dados, a célula para com erro.
# ====================================================================
def url_wyoming(station, date, src, tipo):
    return ('https://weather.uwyo.edu/wsgi/sounding?datetime='
            f'{date:%Y-%m-%d}%20{date:%H:%M:%S}&id={station}&src={src}&type={tipo}')

def le_tabela_wyoming(texto):
    # Tabela TEXT:LIST do Wyoming. As colunas são alinhadas à direita: usa o fim de cada nome do cabeçalho
    # como fim da coluna. Vento: SKNT (nós, formato antigo) ou SPED (m/s, formato atual) -> sempre em nós.
    linhas = re.sub(r'<[^>]+>', '', texto).splitlines()
    i0 = next(i for i, l in enumerate(linhas) if 'PRES' in l.split() and 'HGHT' in l.split())
    cab = linhas[i0]
    nomes = cab.split()
    fins = [m.end() for m in re.finditer(r'\S+', cab)]
    def corta(l):
        out, ini = [], 0
        for fim in fins:
            out.append(l[ini:fim].strip()); ini = fim
        return out
    unidades = dict(zip(nomes, corta(linhas[i0 + 1]))) if i0 + 1 < len(linhas) else {}
    k = i0 + 1
    while k < len(linhas) and (not linhas[k].strip() or linhas[k].strip().startswith('-') or 'hPa' in linhas[k]):
        k += 1
    regs = []
    for l in linhas[k:]:
        if not l.strip() or not re.match(r'^\s*-?\d', l):
            break
        regs.append([float(x) if x else np.nan for x in corta(l)])
    t = pd.DataFrame(regs, columns=nomes)
    if 'SKNT' in t:
        kt = t['SKNT']
    elif 'SPED' in t:
        kt = t['SPED'] * (1.943844 if unidades.get('SPED', 'm/s').lower().startswith('m') else 1.0)
    else:
        raise ValueError('coluna de velocidade do vento (SKNT/SPED) não encontrada')
    return pd.DataFrame({'p': t['PRES'], 'z': t['HGHT'], 'T': t['TEMP'], 'Td': t['DWPT'],
                         'drct': t['DRCT'], 'sknt': kt})

def le_indices_wyoming(texto):
    # Índices calculados pelo Wyoming. Formato atual: tabela <TR><TD>CÓDIGO</TD><TD>Nome</TD><TD>valor</TD><TD>unidade</TD>.
    # Formato antigo: linhas "Nome: valor". Devolve {"Nome (CÓDIGO)": valor}.
    ind = {}
    if not texto:
        return ind
    padrao = (r'<TR>\s*<TD>([^<]*)</TD>\s*<TD>([^<]*)</TD>\s*<TD[^>]*>\s*(-?\d+(?:\.\d+)?)\s*</TD>'
              r'\s*<TD>([^<]*)</TD>')
    for cod, nome, val, unid in re.findall(padrao, texto, flags=re.I | re.S):
        if cod.strip() not in ('SLAT', 'SLON', 'SELV'):
            ind[f'{nome.strip()} ({cod.strip()})'] = float(val)
    if not ind:
        for l in re.sub(r'<[^>]+>', '\n', texto).splitlines():
            m = re.match(r'^\s*([A-Za-z][^:]{1,80}?):\s*(-?\d+(?:\.\d+)?)\s*$', l)
            if m and not m.group(1).strip().startswith(('Station', 'Observation')):
                ind[m.group(1).strip()] = float(m.group(2))
    return ind

def tabela_colab(texto):
    # mesmos nomes de colunas do siphon / notebook original
    return le_tabela_wyoming(texto).rename(columns={'p': 'pressure', 'z': 'height', 'T': 'temperature',
                                                    'Td': 'dewpoint', 'drct': 'direction', 'sknt': 'speed'})

def busca_wyoming(station, date, src):
    # Devolve (df, fonte, índices do Wyoming). Guarda cópia do texto original (arquivo_*), reutilizada se já existir.
    arq_tab = f'wyoming_{station}_{date:%Y%m%d_%H}Z.txt'
    arq_ind = f'wyoming_indices_{station}_{date:%Y%m%d_%H}Z.txt'
    erros = []
    texto = None
    try:
        r = requests.get(url_wyoming(station, date, src, 'TEXT:LIST'), timeout=60); r.raise_for_status()
        tabela_colab(r.text)                            # valida
        texto, fonte = r.text, url_wyoming(station, date, src, 'TEXT:LIST')
        open(arq_tab, 'w').write(texto)
    except Exception as e:
        erros.append(f'wsgi TEXT:LIST: {type(e).__name__}: {e}')
    if texto is None and os.path.exists(arq_tab):
        texto, fonte = open(arq_tab).read(), f'cópia salva {arq_tab} (texto original do Wyoming)'
    if texto is not None:
        df = tabela_colab(texto)
    else:
        try:
            df = WyomingUpperAir.request_data(date, station)
            fonte = 'Wyoming via siphon'
        except Exception as e:
            erros.append(f'siphon: {type(e).__name__}: {e}')
            raise RuntimeError(f'Sem dados de {station} em {date:%Y-%m-%d %HZ}: {erros}')
    ind_txt = None
    try:
        r = requests.get(url_wyoming(station, date, src, 'INDICES'), timeout=60); r.raise_for_status()
        if le_indices_wyoming(r.text):
            ind_txt = r.text; open(arq_ind, 'w').write(ind_txt)
    except Exception as e:
        erros.append(f'wsgi INDICES: {type(e).__name__}: {e}')
    if ind_txt is None and os.path.exists(arq_ind):
        ind_txt = open(arq_ind).read()
    if ind_txt is None and texto and le_indices_wyoming(texto):
        ind_txt = texto                     # a página TEXT:LIST atual também traz a tabela de índices
    if erros:
        print('avisos:', *erros, sep='\n  ')
    return df, fonte, le_indices_wyoming(ind_txt)

def limpa_sondagem(df):
    # Só T, Td e altura são obrigatórios: níveis sem vento (comuns no topo) ficam para os cálculos termodinâmicos
    df = (df.dropna(subset=['pressure', 'temperature', 'dewpoint', 'height'])
            .sort_values('pressure', ascending=False).drop_duplicates('pressure').reset_index(drop=True))
    u, v = mpcalc.wind_components(df['speed'].values * units.knots, df['direction'].values * units.degrees)
    df['u_wind'], df['v_wind'] = u.m, v.m
    return df

def escreve_sounding_txt(df, station, date, arquivo='sounding.txt'):
    # Formato lido pelo wyoming.f: 10 linhas de cabeçalho; dados FORMAT(1X,F6.1,9X,F5.1,2X,F5.1) = p, T, Td;
    # depois as linhas de identificação da estação.
    with open(arquivo, 'w') as f:
        f.write('\n' * 10)
        for _, r in df.iterrows():
            f.write(f"{r['pressure']:7.1f}{r['height']:7.0f}{r['temperature']:7.1f}{r['dewpoint']:7.1f}\n")
        f.write('</PRE>\n')
        f.write(f'Station identifier: {station}\n')
        f.write(f'Station number: {station}\n')
        f.write(f'Observation time: {date:%y%m%d} {date:%H%M}\n')

df, fonte, indices_wyoming = busca_wyoming(station, date, src)
df = limpa_sondagem(df)
escreve_sounding_txt(df, station, date)
print(f'{len(df)} níveis — fonte: {fonte}')
print('Índices calculados pelo Wyoming (guia de conferência):')
display(pd.Series(indices_wyoming, dtype=float) if indices_wyoming else 'indisponíveis')
df

### Skew-T Log-P Diagram

This plot is a Skew-T log-P diagram, a thermodynamic diagram used extensively in meteorology to display atmospheric soundings and analyze atmospheric stability.

*   **Axes:**
    *   The **horizontal axis** represents **temperature**, skewed at an angle (typically 45 degrees) to the isobars (lines of constant pressure).
    *   The **vertical axis** represents the **logarithm of pressure**, increasing downwards.

*   **Lines on the Diagram:**
    *   **Isobars:** Horizontal lines representing constant pressure.
    *   **Isotherms:** Skewed lines representing constant temperature.
    *   **Dry Adiabats:** Lines representing the temperature change of a dry air parcel lifted or lowered adiabatically.
    *   **Moist Adiabats:** Lines representing the temperature change of a saturated air parcel lifted or lowered adiabatically, accounting for the release of latent heat during condensation.
    *   **Mixing Ratio Lines:** Lines representing constant mixing ratio (the mass of water vapor per unit mass of dry air).

*   **Plotted Data:**
    *   The **red line** typically represents the **air temperature profile** with height (or pressure).
    *   The **green line** typically represents the **dewpoint temperature profile** with height (or pressure).
    *   The **black line** represents the **parcel trace**, showing the temperature and pressure of a lifted air parcel (often a surface parcel, but other levels can be chosen).

*   **Interpretation:**
    *   The separation between the temperature and dewpoint profiles indicates the amount of moisture in the atmosphere. A large separation means dry air, while a small separation means moist air.
    *   The relationship between the environmental temperature profile and the dry and moist adiabats indicates atmospheric stability.
        *   If the environmental temperature profile is to the right of a dry adiabat, the layer is absolutely unstable for dry ascent.
        *   If the environmental temperature profile is to the left of a dry adiabat but to the right of a moist adiabat, the layer is conditionally unstable.
        *   If the environmental temperature profile is to the left of a moist adiabat, the layer is absolutely stable for saturated ascent.
    *   The **Convective Available Potential Energy (CAPE)** is the area on the Skew-T where the lifted parcel temperature is warmer than the environmental temperature (positive buoyancy). It represents the potential energy available for convection.
    *   The **Convective Inhibition (CIN)** is the area where the lifted parcel temperature is colder than the environmental temperature (negative buoyancy) below the Level of Free Convection (LFC). It represents the energy barrier that must be overcome for convection to occur.
    *   Other features like the **Lifting Condensation Level (LCL)**, **Level of Free Convection (LFC)**, and **Equilibrium Level (EL)** can be identified on the diagram and provide further insights into the potential for thunderstorm development.

In [ ]:
# ====================================================================
# Passo 1: Dados (já baixados na célula de busca — sem dados de exemplo)
# ====================================================================
p = df['pressure'].values * units.hPa
T = df['temperature'].values * units.degC
Td = df['dewpoint'].values * units.degC
z = df['height'].values * units.meter
h = z - z[0]                                     # altura acima do solo (AGL)
wind_speed = df['speed'].values * units.knots
wind_dir = df['direction'].values * units.degrees
u, v = mpcalc.wind_components(wind_speed, wind_dir)
w = (df['speed'].notna() & df['direction'].notna()).values   # níveis COM vento -> cálculos cinemáticos
pw, uw, vw, hw = p[w], u[w], v[w], h[w]

# ====================================================================
# Passo 2: Índices (se algum falhar, aparece como n/d e o erro é mostrado)
# ====================================================================
erros_calc = {}
def tenta(nome, f):
    try:
        return f()
    except Exception as e:
        erros_calc[nome] = f'{type(e).__name__}: {e}'
        return None
nd = lambda q: q is None or (hasattr(q, 'magnitude') and np.isnan(np.asarray(q.magnitude)).any())

parcel_prof_sb = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
cape_sb, cin_sb = mpcalc.surface_based_cape_cin(p, T, Td)
cape_ml, cin_ml = mpcalc.mixed_layer_cape_cin(p, T, Td, depth=100 * units.hPa)   # θ e r médios nos 100 hPa inferiores
cape_mu, cin_mu = mpcalc.most_unstable_cape_cin(p, T, Td)
p_mu, T_mu, Td_mu, i_mu = mpcalc.most_unstable_parcel(p, T, Td)
parcel_prof_mu = mpcalc.parcel_profile(p[i_mu:], T_mu, Td_mu).to('degC')

lcl_sb = mpcalc.lcl(p[0], T[0], Td[0])
lfc_sb = mpcalc.lfc(p, T, Td, parcel_prof_sb)
el_sb = mpcalc.el(p, T, Td, parcel_prof_sb)
pwat = mpcalc.precipitable_water(p, Td)

li = tenta('LI', lambda: mpcalc.lifted_index(p, T, parcel_prof_sb)[0])   # 3º argumento = perfil da parcela
k_index = tenta('K', lambda: mpcalc.k_index(p, T, Td))
showalter = tenta('Showalter', lambda: mpcalc.showalter_index(p, T, Td)[0])
sweat = tenta('SWEAT', lambda: mpcalc.sweat_index(p[w], T[w], Td[w], wind_speed[w], wind_dir[w])[0])
total_totals = tenta('TT', lambda: mpcalc.total_totals_index(p, T, Td))
dcape = tenta('DCAPE', lambda: mpcalc.downdraft_cape(p, T, Td)[0])

shear_0_1km = mpcalc.bulk_shear(pw, uw, vw, height=hw, depth=1 * units.km)
shear_0_3km = mpcalc.bulk_shear(pw, uw, vw, height=hw, depth=3 * units.km)
shear_0_6km = mpcalc.bulk_shear(pw, uw, vw, height=hw, depth=6 * units.km)
bs = lambda s: mpcalc.wind_speed(s[0], s[1]).to('kt')

# Movimento de tempestade de Bunkers. No Hemisfério Sul a supercélula favorecida é a LEFT-MOVER (LM)
# e a helicidade ciclônica é NEGATIVA.
bunkers_rm, bunkers_lm, vento_medio = mpcalc.bunkers_storm_motion(pw, uw, vw, hw)
srh_0_1km = mpcalc.storm_relative_helicity(hw, uw, vw, depth=1*units.km, storm_u=bunkers_lm[0], storm_v=bunkers_lm[1])[2]
srh_0_3km = mpcalc.storm_relative_helicity(hw, uw, vw, depth=3*units.km, storm_u=bunkers_lm[0], storm_v=bunkers_lm[1])[2]
srh_0_3km_rm = mpcalc.storm_relative_helicity(hw, uw, vw, depth=3*units.km, storm_u=bunkers_rm[0], storm_v=bunkers_rm[1])[2]

# BRN (Weisman & Klemp 1982): CAPE / [0.5 |V(0-6 km) - V(0-500 m)|²], ventos médios nas camadas
def vento_medio_camada(topo):
    m = hw <= topo
    return np.mean(uw[m]), np.mean(vw[m])
u6, v6 = vento_medio_camada(6 * units.km); u05, v05 = vento_medio_camada(500 * units.m)
brn_shear = 0.5 * ((u6 - u05)**2 + (v6 - v05)**2)
brn = (cape_mu / brn_shear).to('dimensionless') if brn_shear.magnitude > 0 else np.nan * units.dimensionless

# Compostos adaptados ao HS (SRH ciclônica negativa -> usa -SRH)
ehi = (cape_sb * (-srh_0_3km) / (160000 * units('m^4/s^4'))).to('dimensionless')
lcl_h = np.interp(np.log(lcl_sb[0].m), np.log(p.m[::-1]), h.m[::-1])
t_lcl = 1.0 if lcl_h < 1000 else (0.0 if lcl_h > 2000 else (2000 - lcl_h) / 1000)
bs6 = bs(shear_0_6km).to('m/s').m
t_bs = 0.0 if bs6 < 12.5 else min(bs6, 30.0) / 20
stp = (cape_sb.to('J/kg').m / 1500) * t_lcl * ((-srh_0_1km).to('m^2/s^2').m / 150) * t_bs   # STP camada fixa (Thompson et al. 2012)

baixo = df[(df['pressure'] >= 700) & df['speed'].notna()]; k_max = baixo['speed'].idxmax()
vento_max_txt = f"{baixo.loc[k_max,'speed']:.0f} kt de {baixo.loc[k_max,'direction']:.0f}° em {baixo.loc[k_max,'pressure']:.0f} hPa"

# ====================================================================
# Passo 3: Figura com Skew-T e Hodógrafo
# ====================================================================
fig = plt.figure(figsize=(16, 9))
skew = SkewT(fig, rotation=45, subplot=(1, 2, 1))
ax_hodo = fig.add_subplot(1, 2, 2)

skew.plot(p, T, 'r', linewidth=2, label='Temperatura')
skew.plot(p, Td, 'g', linewidth=2, label='Ponto de Orvalho')
skew.plot(p, parcel_prof_sb, 'k--', linewidth=1, label='Parcela Superfície')
if p_mu < p[0]:
    skew.plot(p[i_mu:], parcel_prof_mu, color='purple', ls=':', lw=1.5, label=f'Parcela MU ({p_mu:.0f~P})')
skew.plot_barbs(pw[::2], uw[::2], vw[::2])
skew.plot_dry_adiabats(color='brown', linestyle='--', linewidth=0.5)
skew.plot_moist_adiabats(color='blue', linestyle='--', linewidth=0.5)
skew.plot_mixing_lines(color='gray', linestyle='--', linewidth=0.5)
skew.shade_cape(p, T, parcel_prof_sb, color='lightcoral', alpha=0.4)   # sombreia com o PERFIL DA PARCELA
skew.shade_cin(p, T, parcel_prof_sb, color='skyblue', alpha=0.4)
if lcl_sb[0]: skew.ax.axhline(lcl_sb[0], color='purple', linestyle='--', linewidth=1, label=f'LCL: {lcl_sb[0]:.0f~P}')
if not nd(lfc_sb[0]): skew.ax.axhline(lfc_sb[0], color='orange', linestyle='--', linewidth=1, label=f'LFC: {lfc_sb[0]:.0f~P}')
if not nd(el_sb[0]): skew.ax.axhline(el_sb[0], color='red', linestyle='--', linewidth=1, label=f'EL: {el_sb[0]:.0f~P}')
skew.ax.set_ylim(1050, 100)
skew.ax.set_xlim(-40, 45)
skew.ax.set_title("Skew-T / Log-P", fontsize=14)
skew.ax.set_xlabel('Temperatura (°C)'); skew.ax.set_ylabel('Pressão (hPa)')
skew.ax.legend(fontsize=8)

# Hodógrafo (alturas OBSERVADAS acima do solo)
sel = pw >= 100 * units.hPa
h_km = hw.to('km').m
hodo = Hodograph(ax_hodo, component_range=80.)
hodo.add_grid(increment=20, color='gray', linestyle='--')
l = hodo.plot_colormapped(uw[sel], vw[sel], h_km[sel], cmap='viridis')
plt.colorbar(l, ax=ax_hodo, label='altura acima do solo (km)', shrink=0.8)
for p_level in [1000, 925, 850, 700, 500, 300, 200]:
    k = np.argmin(np.abs(pw.m - p_level))
    if abs(pw.m[k] - p_level) < 5:
        ax_hodo.text(uw[k].m, vw[k].m, f'{p_level} hPa ({h_km[k]:.1f} km)', fontsize=9, ha='center', va='bottom')
for nome, vec, mk, cor in (('LM', bunkers_lm, 's', 'tab:blue'), ('RM', bunkers_rm, 'D', 'tab:red')):
    ax_hodo.plot(vec[0].to('kt').m, vec[1].to('kt').m, mk, color=cor, ms=8, label=f'Bunkers {nome}')
ax_hodo.legend(loc='lower left', fontsize=9)
ax_hodo.set_title("Hodógrafo com Níveis de Pressão e Altura", fontsize=14)
ax_hodo.set_xlabel("Componente U (knots)")
ax_hodo.set_ylabel("Componente V (knots)")

# ====================================================================
# Passo 4: Tabela de índices (27 itens, mesma ordem usada na explicação abaixo)
# ====================================================================
fmt = lambda q, f: 'n/d' if nd(q) else format(q, f)
all_indices = [
    f"SBCAPE: {cape_sb:.0f~P}", f"SBCIN: {cin_sb:.0f~P}", f"MLCAPE: {cape_ml:.0f~P}",
    f"MLCIN: {cin_ml:.0f~P}", f"MUCAPE: {cape_mu:.0f~P}", f"MUCIN: {cin_mu:.0f~P}",
    f"PWAT: {pwat:.1f~P}", f"LCL: {lcl_sb[0]:.0f~P}", f"LFC: {fmt(lfc_sb[0], '.0f~P')}",
    f"EL: {fmt(el_sb[0], '.0f~P')}",
    f"Lifted Index: {'n/d' if nd(li) else f'{li.m:.1f} K'}", f"K Index: {'n/d' if nd(k_index) else f'{k_index.m:.1f} °C'}",
    f"Showalter: {'n/d' if nd(showalter) else f'{showalter.m:.1f} K'}", f"SWEAT: {'n/d' if nd(sweat) else f'{sweat.m:.0f}'}",
    f"Shear 0-1 km: {bs(shear_0_1km):.1f~P}", f"Shear 0-3 km: {bs(shear_0_3km):.1f~P}", f"Shear 0-6 km: {bs(shear_0_6km):.1f~P}",
    f"Bunkers LM: ({bunkers_lm[0].to('m/s').m:.1f}, {bunkers_lm[1].to('m/s').m:.1f}) m/s",
    f"SRH 0-1 km (LM): {srh_0_1km:.0f~P}", f"SRH 0-3 km (LM): {srh_0_3km:.0f~P}", f"SRH 0-3 km (RM): {srh_0_3km_rm:.0f~P}",
    f"BRN: {'n/d' if nd(brn) else f'{brn.m:.1f}'}", f"EHI 0-3 (HS): {ehi.m:.2f}", f"DCAPE: {fmt(dcape, '.0f~P')}",
    f"STP fixo (HS): {stp:.2f}", f"Total Totals: {'n/d' if nd(total_totals) else f'{total_totals.m:.1f} °C'}", f"Vento máx <700 hPa: {vento_max_txt}"
]
table_data = np.reshape(all_indices, (3, 9)).tolist()
ax_table = fig.add_axes([0, 0, 1, 0.2])
ax_table.axis('off')
table = ax_table.table(cellText=table_data, loc='center', cellLoc='left', edges='closed')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.5)

fig.suptitle(f"Análise de Sondagem para {station} - {date.strftime('%Y-%m-%d %H:%M')} UTC", fontsize=18)
fig.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.2)
plt.show()
if erros_calc:
    print('⚠ índices que falharam:', erros_calc)

In [ ]:
# Tabela de índices em figura separada (mesmos valores da figura acima)
fig_table, ax_table = plt.subplots(figsize=(14, 2.2))
ax_table.axis('off')
table = ax_table.table(cellText=np.reshape(all_indices, (3, 9)).tolist(), loc='center', cellLoc='left', edges='closed')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.5)
plt.title("Tabela de Índices")
plt.show()

### Diagrama Skew-T Log-P

Este gráfico é um diagrama Skew-T log-P, um diagrama termodinâmico amplamente utilizado em meteorologia para exibir sondagens atmosféricas e analisar a estabilidade atmosférica.

* **Eixos:**
  * O **eixo horizontal** representa a **temperatura**, inclinada em um ângulo (geralmente 45 graus) em relação às isóbaras (linhas de pressão constante).
  * O **eixo vertical** representa o **logaritmo da pressão**, aumentando para baixo.
* **Linhas no Diagrama:**
  * **Isóbaras:** Linhas horizontais representando pressão constante.
  * **Isotermas:** Linhas inclinadas representando temperatura constante.
  * **Adiabáticas Secas:** Linhas representando a mudança de temperatura de uma parcela de ar seco levantada ou abaixada adiabaticamente.
  * **Adiabáticas Úmidas:** Linhas representando a mudança de temperatura de uma parcela de ar saturado levantada ou abaixada adiabaticamente, considerando a liberação de calor latente durante a condensação.
  * **Linhas de Razão de Mistura:** Linhas representando razão de mistura constante (a massa de vapor d'água por unidade de massa de ar seco).
* **Dados Plotados:**
  * A **linha vermelha** geralmente representa o **perfil de temperatura do ar** com a altura (ou pressão).
  * A **linha verde** geralmente representa o **perfil de temperatura do ponto de orvalho** com a altura (ou pressão).
  * A **linha preta** representa o **traçado da parcela**, mostrando a temperatura e pressão de uma parcela de ar levantada (geralmente uma parcela de superfície, mas outros níveis podem ser escolhidos).
* **Interpretação:**
  * A separação entre os perfis de temperatura e ponto de orvalho indica a quantidade de umidade na atmosfera. Uma grande separação significa ar seco, enquanto uma pequena separação significa ar úmido.
  * A relação entre o perfil de temperatura ambiental e as adiabáticas secas e úmidas indica a estabilidade atmosférica.
    * Se o perfil de temperatura ambiental estiver à direita de uma adiabática seca, a camada é absolutamente instável para ascensão seca.
    * Se o perfil de temperatura ambiental estiver à esquerda de uma adiabática seca, mas à direita de uma adiabática úmida, a camada é condicionalmente instável.
    * Se o perfil de temperatura ambiental estiver à esquerda de uma adiabática úmida, a camada é absolutamente estável para ascensão saturada.
  * A **Energia Potencial de Convecção Disponível (CAPE)** é a área no Skew-T onde a temperatura da parcela levantada é mais quente que a temperatura ambiental (flutuabilidade positiva). Ela representa a energia potencial disponível para convecção.
  * A **Inibição Convectiva (CIN)** é a área onde a temperatura da parcela levantada é mais fria que a temperatura ambiental (flutuabilidade negativa) abaixo do Nível de Convecção Livre (LFC). Ela representa a barreira de energia que deve ser superada para que a convecção ocorra.
  * Outras características como o **Nível de Condensação por Levantamento (LCL)**, **Nível de Convecção Livre (LFC)** e **Nível de Equilíbrio (EL)** podem ser identificadas no diagrama e fornecem insights adicionais sobre o potencial de desenvolvimento de tempestades.

In [ ]:
all_indices

In [ ]:
# @title
from IPython.display import Markdown
import numpy as np

# Função segura para obter o valor formatado ou 'N/A'
def get_val(index_list, idx):
    if idx < len(index_list):
        try:
            return index_list[idx].split(': ')[1]
        except IndexError:
            return "N/A"
    return "N/A"

#all_indices_para_uso = all_indices_for_table if 'all_indices_for_table' in globals() else all_indices
# Vamos garantir compatibilidade com as duas listas criadas anteriormente
lista_indices = all_indices

markdown_text = f"""
## Explicação dos Índices Meteorológicos e Valores de Referência

A variável `all_indices` contém uma lista de índices meteorológicos calculados a partir dos dados da sondagem atmosférica. Esses índices são utilizados para avaliar a estabilidade da atmosfera, o potencial para tempo severo e outras condições meteorológicas. Abaixo, apresentamos a explicação de cada um deles, os valores calculados para a sondagem atual e alguns valores de referência para análise:

* **SBCAPE (Surface-Based Convective Available Potential Energy):** Energia potencial disponível para convecção baseada em uma parcela de ar levantada a partir da superfície. Valores altos indicam maior potencial para tempestades.
  * **Valor Calculado:** {get_val(lista_indices, 0)}
  * **Valores de Referência:**
    * Baixo: < 500 J/kg
    * Moderado: 500 - 1500 J/kg
    * Alto: 1500 - 3000 J/kg
    * Muito Alto: > 3000 J/kg

* **SBCIN (Surface-Based Convective Inhibition):** Energia que impede a ascensão de uma parcela de ar a partir da superfície. Valores negativos (ou próximos de zero) indicam pouca ou nenhuma inibição para o desenvolvimento de convecção.
  * **Valor Calculado:** {get_val(lista_indices, 1)}
  * **Valores de Referência:**
    * Significativa: < -100 J/kg
    * Moderada: -100 a -25 J/kg
    * Fraca: -25 a 0 J/kg
    * Nula ou Positiva: >= 0 J/kg

* **MLCAPE (Mixed-Layer Convective Available Potential Energy):** CAPE calculada para uma parcela média da camada mais baixa da atmosfera (geralmente os primeiros 50-100 hPa acima da superfície). Representa o potencial de convecção para movimentos ascendentes que se originam de uma camada mais profunda.
  * **Valor Calculado:** {get_val(lista_indices, 2)}
  * **Valores de Referência:** Similares aos do SBCAPE, mas mais representativo para convecção iniciada em camadas mais profundas.

* **MLCIN (Mixed-Layer Convective Inhibition):** CIN calculada para a mesma parcela média da camada misturada.
  * **Valor Calculado:** {get_val(lista_indices, 3)}
  * **Valores de Referência:** Similares aos do SBCIN, mas mais representativo para convecção iniciada em camadas mais profundas.

* **MUCAPE (Most Unstable Convective Available Potential Energy):** O maior valor de CAPE encontrado ao levantar parcelas a partir de todos os níveis na sondagem. Indica o maior potencial de energia disponível para convecção, independentemente do nível de origem.
  * **Valor Calculado:** {get_val(lista_indices, 4)}
  * **Valores de Referência:** Similares aos do SBCAPE, mas indica o potencial máximo de energia disponível.

* **MUCIN (Most Unstable Convective Inhibition):** CIN associado à parcela mais instável (aquela que produziu a MUCAPE).
  * **Valor Calculado:** {get_val(lista_indices, 5)}
  * **Valores de Referência:** Similares aos do SBCIN, associado ao potencial máximo de convecção.

* **PWAT (Precipitable Water):** Quantidade total de vapor d'água em uma coluna vertical da atmosfera. Representa o potencial máximo de precipitação. Valores altos indicam uma atmosfera muito úmida.
  * **Valor Calculado:** {get_val(lista_indices, 6)}
  * **Valores de Referência:** Variam sazonal e geograficamente. Valores acima da média climática local indicam maior umidade disponível para precipitação.

* **LCL (Lifting Condensation Level):** Nível onde uma parcela de ar levantada se torna saturada e a condensação começa a ocorrer (formação de nuvens).
  * **Valor Calculado:** {get_val(lista_indices, 7)}
  * **Valores de Referência:** LCLs mais baixos (pressões mais altas) indicam que as nuvens se formarão mais perto da superfície, o que pode favorecer o desenvolvimento de tornados em ambientes com cisalhamento adequado.

* **LFC (Level of Free Convection):** Nível onde uma parcela de ar levantada se torna mais quente que o ambiente e começa a ascender livremente devido à flutuabilidade positiva.
  * **Valor Calculado:** {get_val(lista_indices, 8)}
  * **Valores de Referência:** A existência de um LFC abaixo do EL indica potencial para convecção. A altura do LFC influencia a profundidade da camada de CIN.

* **EL (Equilibrium Level):** Nível onde a temperatura de uma parcela de ar ascendente se iguala à temperatura do ambiente novamente, e a flutuabilidade se torna neutra ou negativa. Frequentemente associado ao topo das nuvens de tempestade.
  * **Valor Calculado:** {get_val(lista_indices, 9)}
  * **Valores de Referência:** A altura do EL indica até onde as correntes ascendentes podem se estender. ELs mais altos estão associados a tempestades mais intensas e potencial para granizo maior.

* **Lifted Index:** Índice simples de estabilidade: temperatura do ambiente menos a da parcela de superfície levantada até 500 hPa. Valores negativos indicam instabilidade.
  * **Valor Calculado:** {get_val(lista_indices, 10)}
  * **Valores de Referência:**
    * > 0: Estável
    * 0 a -2: Ligeiramente Instável
    * -2 a -6: Moderadamente Instável
    * < -6: Muito Instável

* **K Index:** Índice que avalia o potencial de desenvolvimento de tempestades não severas baseadas em umidade e gradientes de temperatura em diferentes níveis. Valores mais altos indicam maior potencial para chuvas e tempestades.
  * **Valor Calculado:** {get_val(lista_indices, 11)}
  * **Valores de Referência:**
    * < 15: Sem Potencial para Tempestades
    * 15 - 20: Pequeno Potencial
    * 20 - 25: Potencial Moderado para Chuvas e Tempestades
    * 25 - 30: Potencial Moderado para Tempestades
    * 30 - 35: Potencial para Tempestades Generalizadas
    * > 35: Potencial para Tempestades Fortes

* **Showalter Index:** Índice de estabilidade calculado levantando-se uma parcela a partir do nível de 850 hPa até 500 hPa. Semelhante ao Lifted Index, valores negativos indicam instabilidade.
  * **Valor Calculado:** {get_val(lista_indices, 12)}
  * **Valores de Referência:**
    * > +4: Estável
    * +1 a +4: Ligeiramente Instável
    * -2 a +1: Moderadamente Instável
    * < -2: Muito Instável

* **SWEAT (Severe Weather Threat) Index:** Índice composto que incorpora temperatura, umidade, cisalhamento do vento e ventos de altos níveis para avaliar o potencial de tempo severo, incluindo tornados. Valores mais altos indicam maior potencial.
  * **Valor Calculado:** {get_val(lista_indices, 13)}
  * **Valores de Referência:**
    * < 250: Baixo Potencial para Tempo Severo
    * 250 - 300: Potencial Moderado para Tempo Severo
    * > 300: Alto Potencial para Tempo Severo

* **Shear 0-1 km:** Diferença vetorial do vento entre a superfície e 1 km acima do solo.
  * **Valor Calculado:** {get_val(lista_indices, 14)}
  * **Valores de Referência:** Baixo < 10 kt; Moderado 10–20 kt; Alto > 20 kt.

* **Shear 0-3 km:** Diferença vetorial do vento entre a superfície e 3 km.
  * **Valor Calculado:** {get_val(lista_indices, 15)}
  * **Valores de Referência:** Baixo < 15 kt; Moderado 15–30 kt; Alto > 30 kt.

* **Shear 0-6 km:** Diferença vetorial do vento entre a superfície e 6 km — principal indicador de organização da convecção.
  * **Valor Calculado:** {get_val(lista_indices, 16)}
  * **Valores de Referência:** Baixo < 20 kt; Moderado 20–40 kt; Alto > 40 kt (supercélulas).

* **Bunkers LM (left-mover):** Movimento estimado da tempestade que se desvia à esquerda do vento médio (Bunkers et al. 2000).
  No **Hemisfério Sul** é a supercélula favorecida quando o hodógrafo gira no sentido anti-horário com a altura.
  * **Valor Calculado:** {get_val(lista_indices, 17)}

* **SRH 0-1 km e 0-3 km (LM):** Helicidade relativa à tempestade, SRH = ∫ (V − c) · (k × ∂V/∂z) dz, com c = movimento LM.
  No **Hemisfério Sul a helicidade ciclônica é NEGATIVA**: compare o **módulo** dos valores negativos com as referências abaixo (do HN).
  * **Valor Calculado 0-1 km:** {get_val(lista_indices, 18)}
  * **Valor Calculado 0-3 km:** {get_val(lista_indices, 19)}
  * **Valores de Referência (|SRH|):** 0-1 km: alto > 100 m²/s²; 0-3 km: moderado 150–250, alto > 250 m²/s².

* **SRH 0-3 km (RM):** Mesma conta com o movimento right-mover — para comparação (no HS costuma ser positiva/fraca).
  * **Valor Calculado:** {get_val(lista_indices, 20)}

* **BRN (Bulk Richardson Number):** MUCAPE / [½ |V(0–6 km) − V(0–500 m)|²] (Weisman & Klemp 1982), com ventos médios nas camadas.
  * **Valor Calculado:** {get_val(lista_indices, 21)}
  * **Valores de Referência:** 10–45 favorece supercélulas; > 50 favorece multicélulas.

* **EHI 0-3 km (HS):** SBCAPE × (−SRH 0-3 km LM) / 160000.
  * **Valor Calculado:** {get_val(lista_indices, 22)}
  * **Valores de Referência:** > 1 favorável a supercélulas; > 2 potencial significativo.

* **DCAPE (Downdraft CAPE, MetPy):** Energia disponível para correntes descendentes (parcela saturada no mínimo de θe dos 400 hPa inferiores, descendo até a superfície).
  * **Valor Calculado:** {get_val(lista_indices, 23)}
  * **Valores de Referência:** > 1000 J/kg favorece rajadas fortes.
  *(o programa de Emanuel calcula um DCAPE com outra definição — ver a seção do Fortran)*

* **STP de camada fixa (HS):** (SBCAPE/1500) · ((2000 − LCL)/1000) · (−SRH₀₋₁ LM /150) · (Shear₀₋₆/20), com limites de Thompson et al. (2012).
  * **Valor Calculado:** {get_val(lista_indices, 24)}
  * **Valores de Referência:** > 1 associado a tornados significativos (climatologia dos EUA).

* **Total Totals:** (T850 − T500) + (Td850 − T500).
  * **Valor Calculado:** {get_val(lista_indices, 25)}
  * **Valores de Referência:** > 44 tempestades possíveis; > 50 tempestades severas possíveis.

* **Vento máximo abaixo de 700 hPa:** Indicador de jato em baixos níveis (JBN) — velocidade, direção e nível.
  * **Valor Calculado:** {get_val(lista_indices, 26)}
"""

display(Markdown(markdown_text))

### Conferência com os índices calculados pelo Wyoming

O Wyoming publica seus próprios índices para cada sondagem. Eles servem de **guia**: diferenças pequenas são normais
(temperatura virtual, outra parcela — o LCL do Wyoming é de uma parcela média dos níveis baixos —, outra integração);
diferenças grandes devem ser investigadas. Os números usados na apresentação são sempre os **calculados aqui**.

In [ ]:
calc = {'SHOW': showalter, 'LFVT': li, 'LIFT': li, 'SWET': sweat, 'KINX': k_index, 'TOTL': total_totals,
        'DCAPE': dcape, 'MUCAPE': cape_mu, 'MUCIN': cin_mu, 'LCLP': lcl_sb[0], 'PWAT': pwat}
linhas = []
for nome, val in indices_wyoming.items():
    m = re.search(r'\(([A-Z]+)\)\s*$', nome)
    q = calc.get(m.group(1)) if m else None
    v = None if q is None or nd(q) else float(np.asarray(q.m))
    linhas.append(dict(indice_wyoming=nome, valor_wyoming=val, calculado_aqui=v,
                       diferenca=None if v is None else v - val))
display(pd.DataFrame(linhas).round(1) if linhas else 'índices do Wyoming indisponíveis')



---
# AGORA iremos fazer outra analises sobre essa mesma sondagem


In [ ]:
#Adaptando as variaveis para usar códigos do Matlab
p = df['pressure'].values * units.hPa
T = df['temperature'].values * units.degC
Td = df['dewpoint'].values * units.degC
u = df['u_wind'].values * units.knots
v = df['v_wind'].values * units.knots
zz = df['height'].values * units.meter            # altura OBSERVADA (antes: atmosfera padrão)
theta=mpcalc.potential_temperature(p,T)
thetae=mpcalc.equivalent_potential_temperature(p, T, Td)
rh=mpcalc.relative_humidity_from_dewpoint(T,Td)
mixrat=mpcalc.mixing_ratio_from_relative_humidity(p,T,rh)
thetas=mpcalc.saturation_equivalent_potential_temperature(p,T)   # θes
pw = mpcalc.precipitable_water(p, Td)
S = -(T.to('K').m / theta.m) * np.gradient(theta.m, p.m)          # S = -(T/θ) ∂θ/∂p  [K/hPa]
N2 = mpcalc.brunt_vaisala_frequency_squared(zz, theta).to('1/s^2').m   # N² (negativo = camada instável)
N = np.sqrt(np.where(N2 > 0, N2, np.nan))
P_TOPO = 300   # topo dos gráficos de perfil (hPa)
pw # precipitable Water!!

In [ ]:
data = {
    'Pressure (hPa)': p.magnitude,
    'Height (m)': zz.magnitude,
    'Temperature (degC)': T.magnitude,
    'Dewpoint (degC)': Td.magnitude,
    'U Wind (knots)': u.magnitude,
    'V Wind (knots)': v.magnitude,
    'Potential Temperature (K)': theta.magnitude,
    'Equivalent Potential Temperature (K)': thetae.magnitude,
    'Relative Humidity (%)': rh.to('percent').magnitude,
    'Mixing Ratio (g/kg)': mixrat.to('g/kg').magnitude,
    'Saturation Equivalent Potential Temperature (K)': thetas.magnitude,
    'Static Stability S (K/hPa)': S,
    'N² (1/s²)': N2,
    'Brunt-Vaisala Frequency N (1/s)': N,
}
df_calculated = pd.DataFrame(data)
display(df_calculated)

### Gráfico de Temperatura Potencial, Temperatura Potencial Equivalente e Temperatura Potencial de Saturação

Este gráfico exibe três perfis de temperatura diferentes em função da pressão:

*   **Temperatura Potencial ($\theta$)**: A temperatura que uma parcela de ar seco teria se fosse movida adiabaticamente para uma pressão padrão de 1000 hPa. É conservada durante processos adiabáticos secos.
*   **Temperatura Potencial Equivalente ($\theta_e$)**: A temperatura que uma parcela de ar teria se toda a sua umidade fosse condensada e o calor latente liberado, e a parcela fosse então movida adiabaticamente para uma pressão padrão de 1000 hPa. É conservada durante processos adiabáticos secos e úmidos.
*   **Temperatura Potencial de Saturação ($\theta_s$)**: A temperatura potencial de uma parcela de ar saturado.
*   **Interpretação:**
    *   O perfil vertical da temperatura potencial indica **estabilidade estática** em ar seco. Se a temperatura potencial aumenta com a altura, a camada é estável em relação a deslocamentos adiabáticos secos.
    *   O perfil vertical da temperatura potencial equivalente indica **estabilidade estática** em ar saturado. Se a temperatura potencial equivalente aumenta com a altura, a camada é estável em relação a deslocamentos adiabáticos úmidos.
    *   Comparar os perfis de $\theta$, $\theta_e$ e $\theta_s$ pode fornecer insights sobre a estabilidade atmosférica e o potencial para convecção. Por exemplo, se $\theta_e$ diminui com a altura, a atmosfera é condicionalmente instável.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(theta, p, label='θ (temperatura potencial)')
ax.plot(thetae, p, label='θe (temperatura potencial equivalente)')
ax.plot(thetas, p, label='θes (temp. potencial equivalente de saturação)')
ax.set_xlabel('Temperatura (K)')
ax.set_ylabel('Pressão (hPa)')
ax.set_title(f'Temperaturas potenciais\nEstação: {station}, Data: {date.strftime("%Y-%m-%d %H:%M")} UTC')

ax.set_yscale('log')
ax.set_ylim(1050, P_TOPO)
yticks = [t for t in [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 100] if t >= P_TOPO]
ax.set_yticks(yticks)
ax.set_yticklabels([str(y) for y in yticks])
ax.minorticks_off()

sel = p.m >= P_TOPO
ax.set_xlim(np.nanmin(theta.m[sel]) - 5, np.nanmax(thetas.m[sel]) + 5)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Gráfico da Frequência de Brunt-Vaisala

Este gráfico mostra o **quadrado da frequência de Brunt-Väisälä (N²)** em função da pressão, calculado com a altura observada. Usamos N² porque N só existe onde N² > 0: **N² < 0 indica camada estaticamente instável**. A frequência de Brunt-Vaisala é uma medida da frequência com que uma parcela de ar deslocada verticalmente irá oscilar em uma atmosfera estavelmente estratificada.

*   **Interpretação:**
    *   **Valores mais altos de N** indicam uma atmosfera **mais estável** e oscilações mais fortes. Isso significa que a atmosfera é mais resistente ao movimento vertical.
    *   **Valores mais baixos de N** indicam uma atmosfera **menos estável** e oscilações mais fracas.
    *   Em uma atmosfera **instável**, a frequência de Brunt-Vaisala é **imaginária**, e as parcelas acelerarão para longe de sua posição original em vez de oscilar. O gráfico mostra apenas valores reais de N, que correspondem a condições estáveis.

A frequência de Brunt-Vaisala está relacionada à estabilidade estática (S), com maior estabilidade estática correspondendo a maiores frequências de Brunt-Vaisala em condições estáveis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(N2 * 1e4, p, 'k', label='N²')
ax.axvline(0, color='r', lw=0.8)
ax.set_xlabel('N² (10⁻⁴ s⁻²)')
ax.set_ylabel('Pressão (hPa)')
ax.set_title(f'Frequência de Brunt-Väisälä ao quadrado\nEstação: {station}, Data: {date.strftime("%Y-%m-%d %H:%M")} UTC')

ax.set_yscale('log')
ax.set_ylim(1050, P_TOPO)
yticks = [t for t in [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 100] if t >= P_TOPO]
ax.set_yticks(yticks)
ax.set_yticklabels([str(y) for y in yticks])
ax.minorticks_off()

ax.set_xlim(-5, 10)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Gráfico da Estabilidade Estática

Este gráfico mostra a **Estabilidade Estática (S)** em função da pressão. A estabilidade estática é uma medida de se uma parcela de ar retornará à sua posição original após ser deslocada verticalmente.

*   **Interpretação:**
    *   **Valores positivos de S** indicam uma atmosfera com **estratificação estável**. Uma parcela deslocada experimentará uma força restauradora e retornará ao seu nível original.
    *   **Valores negativos de S** indicam uma atmosfera com **estratificação instável**. Uma parcela deslocada acelerará para longe de seu nível original.
    *   **Valores de S próximos de zero** indicam uma atmosfera com **estratificação neutra**. Uma parcela deslocada permanecerá em seu novo nível.

Valores positivos mais altos de estabilidade estática indicam uma resistência maior ao movimento vertical, enquanto valores negativos indicam condições favoráveis para convecção.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(S, p, 'b', label='S = −(T/θ) ∂θ/∂p')
ax.axvline(0, color='r', lw=0.8)
ax.set_xlabel('Estabilidade estática S (K/hPa)')
ax.set_ylabel('Pressão (hPa)')
ax.set_title(f'Estabilidade estática\nEstação: {station}, Data: {date.strftime("%Y-%m-%d %H:%M")} UTC')

ax.set_yscale('log')
ax.set_ylim(1050, P_TOPO)
yticks = [t for t in [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 100] if t >= P_TOPO]
ax.set_yticks(yticks)
ax.set_yticklabels([str(y) for y in yticks])
ax.minorticks_off()

ax.set_xlim(-0.2, 0.6)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Gráficos de Diferença de Temperatura de Densidade

Estes gráficos de contorno mostram a **Diferença de Temperatura de Densidade (DTD)**, que é uma medida de flutuabilidade. A DTD é calculada para parcelas levantadas de vários níveis de pressão (Pressão de Origem da Parcela no eixo x) para diferentes níveis de pressão (Pressão para a qual a Parcela é Levantada no eixo y).

*   **Interpretação:**
    *   **Valores positivos de DTD** indicam que uma parcela levantada está mais quente e menos densa do que o ambiente circundante naquele nível de pressão, o que significa que ela é flutuante e continuará a subir. Áreas com grandes valores positivos de DTD sugerem potencial para fortes correntes ascendentes e instabilidade.
    *   **Valores negativos de DTD** indicam que uma parcela levantada está mais fria e mais densa do que o ambiente circundante, o que significa que ela é negativamente flutuante e afundará. Áreas com valores negativos de DTD representam camadas estáveis.

Os dois gráficos mostram a DTD calculada usando diferentes suposições:

*   **Diferença de Temperatura de Densidade Pseudo-adiabática Média:** Assume que a parcela levantada segue um processo pseudo-adiabático (toda a água condensada cai imediatamente).
*   **Diferença de Temperatura de Densidade Reversível Média:** Assume que a parcela levantada segue um processo reversível (toda a água condensada permanece com a parcela).

Comparar esses dois gráficos pode fornecer insights sobre o impacto potencial do carregamento de precipitação na flutuabilidade da parcela.


---
---

### Programa em Fortran: Compilação e Execução de wyoming.f

Esta seção do script envolve um programa separado escrito em Fortran, denominado `wyoming.f`.

* **Origem de `wyoming.f`**: O arquivo `wyoming.f` é um arquivo de código-fonte em Fortran fornecido como parte do ambiente deste notebook. Sua origem ou autor específico não são detalhados no próprio notebook, mas, com base em seu nome e uso, ele foi projetado para processar dados de sondagem, provavelmente em um formato compatível com os dados obtidos no site Wyoming Upper Air Soundings.
## Mais detalhes olhar em
https://ocw.mit.edu/courses/12-811-tropical-meteorology-spring-2011/pages/tools/

In [ ]:
# O sounding.txt já foi escrito pela célula de busca, A PARTIR DA MESMA SONDAGEM analisada acima,
# no formato que o wyoming.f lê: 10 linhas de cabeçalho e colunas fixas FORMAT(1X,F6.1,9X,F5.1,2X,F5.1) = p, T, Td.
escreve_sounding_txt(df, station, date)

In [ ]:
!head -15 sounding.txt

In [ ]:
!find -name sounding.txt

In [ ]:
# ====================================================================
# Obter, compilar e preparar o programa de Kerry Emanuel (wyoming.f)
# ====================================================================
FONTES_WYOMING_F = ['https://texmex.mit.edu/pub/emanuel/soundings/wyoming.f',                       # original (MIT)
                    'https://raw.githubusercontent.com/reinaldohaas/tarefa-meso/main/metpack/wyoming.f']  # cópia da disciplina
def obtem_wyoming_f():
    if os.path.exists('wyoming.f') and 'PROGRAM' in open('wyoming.f', errors='ignore').read(2000).upper():
        return 'wyoming.f já presente'
    for url in FONTES_WYOMING_F:
        try:
            r = requests.get(url, timeout=60); r.raise_for_status()
            if 'PROGRAM' in r.text[:2000].upper() and 'SOUNDING.TXT' in r.text.upper():
                open('wyoming.f', 'w').write(r.text)
                return url
        except Exception as e:
            print('  falhou', url, '->', type(e).__name__, e)
    raise RuntimeError('Não consegui obter o wyoming.f. Faça upload do arquivo (metpack/wyoming.f do repositório) '
                       'para a pasta do Colab e rode esta célula de novo.')
print('wyoming.f:', obtem_wyoming_f())

# O wyoming.f original grava as matrizes com MAX(valor,-4.0): tudo abaixo de -4 K vira -4 K.
# CORTE_EMANUEL = False compila uma cópia SEM esse corte (o resto do programa é idêntico).
CORTE_EMANUEL = False
codigo = open('wyoming.f').read()
if not CORTE_EMANUEL:
    linhas_corte = ['TRDBAR(I,J)=MAX(TRDBAR(I,J),-4.0)', 'TPDBAR(I,J)=MAX(TPDBAR(I,J),-4.0)']
    if all(l in codigo for l in linhas_corte):
        for l in linhas_corte:
            codigo = codigo.replace(l, 'CONTINUE')
    else:
        print('⚠ esta versão do wyoming.f não tem as linhas do corte de -4 K; compilando sem alteração')
        CORTE_EMANUEL = True
open('wyoming_usado.f', 'w').write(codigo)

r = subprocess.run(['gfortran', '-O2', '-o', 'wyoming.exe', 'wyoming_usado.f'], capture_output=True, text=True)
if r.returncode != 0 or not os.path.exists('wyoming.exe'):
    raise RuntimeError('Falha ao compilar o wyoming.f:\n' + r.stdout + r.stderr)
print('wyoming.exe compilado')

def roda_wyoming(pasta='.'):
    # Roda o wyoming.exe na pasta (que deve conter sounding.txt) e confere se as saídas foram geradas.
    for f in ('p.out', 'porig.out', 'tdifrev.out', 'tdifpseudo.out', 'cape.out'):
        if os.path.exists(os.path.join(pasta, f)):
            os.remove(os.path.join(pasta, f))            # não reaproveitar saída de outra sondagem
    r = subprocess.run([os.path.abspath('wyoming.exe')], cwd=pasta, capture_output=True, text=True)
    faltando = [f for f in ('p.out', 'porig.out', 'tdifrev.out', 'tdifpseudo.out', 'cape.out')
                if not os.path.exists(os.path.join(pasta, f))]
    if r.returncode != 0 or faltando:
        cab = open(os.path.join(pasta, 'sounding.txt')).read().splitlines()[:16]
        raise RuntimeError(f'wyoming.exe não gerou {faltando} (código {r.returncode}).\n'
                           f'Saída do Fortran:\n{r.stdout}\n{r.stderr}\n'
                           'Início do sounding.txt:\n' + '\n'.join(cab))
    return r.stdout

In [ ]:
escreve_sounding_txt(df, station, date)     # garante que o sounding.txt é o da sondagem atual
print(roda_wyoming('.') or 'wyoming.exe executado — saídas p.out, porig.out, tdifrev.out, tdifpseudo.out, cape.out')

In [ ]:
# np.loadtxt lê todas as linhas (pd.read_fwf sem names usava a 1ª linha de dados como cabeçalho)
pp = np.loadtxt('p.out')                 # níveis para onde a parcela é levantada
porig = np.loadtxt('porig.out')          # níveis de origem da parcela
tdifrev = np.loadtxt('tdifrev.out')      # [nível final, origem] reversível (K)
tdifpseudo = np.loadtxt('tdifpseudo.out')# [nível final, origem] pseudoadiabático (K)
modsound = pd.read_csv('modsound.txt', sep=r'\s+', names=['Pr', 'Temp', 'Rh'])

def le_cape_out(arq='cape.out'):
    linhas = [l.split() for l in open(arq) if re.match(r'^\s*-?\d+\.\d', l)]
    return pd.DataFrame([[float(x) for x in l[:8]] for l in linhas],
                        columns=['p_origem', 'PA_rev', 'PA_pse', 'NA_rev', 'NA_pse', 'CAPE_rev', 'CAPE_pse', 'DCAPE'])
cape_emanuel = le_cape_out()
name = station
cape_emanuel

In [ ]:
pr=modsound['Pr'];
tc=modsound['Temp'];
rh=modsound['Rh'];


In [ ]:
X, Y = np.meshgrid(porig, pp)
print(tdifpseudo.shape, tdifrev.shape, X.shape)

In [ ]:
def plota_emanuel(ax, X, Y, campo, amp):
    Z = np.where(Y < X, campo, np.nan)      # só níveis acima da origem têm sentido físico
    cf = ax.contourf(X, Y, Z, levels=np.arange(-amp, amp + 0.25, 0.5), cmap='RdBu_r',
                     vmin=-amp, vmax=amp, extend='both')
    cs = ax.contour(X, Y, Z, levels=[v for v in np.arange(-amp, amp + 0.1, 2) if v != 0], colors='k', linewidths=0.7)
    ax.clabel(cs, fmt='%d', fontsize=8)
    cz = ax.contour(X, Y, Z, levels=[0], colors='k', linewidths=2)
    ax.clabel(cz, fmt='0', fontsize=9)
    ax.set_xlim(X.max(), X.min()); ax.set_ylim(Y.max(), 100)
    ax.set_xlabel('Pressão de origem da parcela (hPa)')
    ax.set_ylabel('Pressão até onde a parcela é levantada (hPa)')
    ax.grid(alpha=0.3, ls=':')
    return cf

# escala simétrica (branco = 0 K) ajustada à flutuabilidade positiva, mínimo ±6 K
vis = (Y < X) & (Y >= 100)
amp = 2 * np.ceil(max(np.nanmax(tdifrev[vis]), np.nanmax(tdifpseudo[vis]), 6) / 2)
fig, ax = plt.subplots(figsize=(7, 6.5))
cf = plota_emanuel(ax, X, Y, tdifpseudo, amp)
fig.colorbar(cf, orientation='horizontal', label='K', ticks=np.arange(-amp, amp + 1, 2))
fig.suptitle('Mean Pseudo-adiabatic Density Temperature Difference (K)\nStation = ' + str(name) + '   ' + str(hh) + 'Z ' + str(dd) + ' ' + str(mm) + ' ' + str(yy), fontsize=12)
plt.savefig('tdadiabatico.png', dpi=150, bbox_inches='tight')   # salvar ANTES de mostrar
plt.show()

In [ ]:
def plota_emanuel(ax, X, Y, campo, amp):
    Z = np.where(Y < X, campo, np.nan)      # só níveis acima da origem têm sentido físico
    cf = ax.contourf(X, Y, Z, levels=np.arange(-amp, amp + 0.25, 0.5), cmap='RdBu_r',
                     vmin=-amp, vmax=amp, extend='both')
    cs = ax.contour(X, Y, Z, levels=[v for v in np.arange(-amp, amp + 0.1, 2) if v != 0], colors='k', linewidths=0.7)
    ax.clabel(cs, fmt='%d', fontsize=8)
    cz = ax.contour(X, Y, Z, levels=[0], colors='k', linewidths=2)
    ax.clabel(cz, fmt='0', fontsize=9)
    ax.set_xlim(X.max(), X.min()); ax.set_ylim(Y.max(), 100)
    ax.set_xlabel('Pressão de origem da parcela (hPa)')
    ax.set_ylabel('Pressão até onde a parcela é levantada (hPa)')
    ax.grid(alpha=0.3, ls=':')
    return cf

# escala simétrica (branco = 0 K) ajustada à flutuabilidade positiva, mínimo ±6 K
vis = (Y < X) & (Y >= 100)
amp = 2 * np.ceil(max(np.nanmax(tdifrev[vis]), np.nanmax(tdifpseudo[vis]), 6) / 2)
fig, ax = plt.subplots(figsize=(7, 6.5))
cf = plota_emanuel(ax, X, Y, tdifrev, amp)
fig.colorbar(cf, orientation='horizontal', label='K', ticks=np.arange(-amp, amp + 1, 2))
fig.suptitle('Mean Reversible Density Temperature Difference (K)\nStation = ' + str(name) + '   ' + str(hh) + 'Z ' + str(dd) + ' ' + str(mm) + ' ' + str(yy), fontsize=12)
plt.savefig('tdreversivel.png', dpi=150, bbox_inches='tight')   # salvar ANTES de mostrar
plt.show()

### Como interpretar as matrizes de Emanuel

**1. O que os gráficos representam.** Cada ponto mostra a diferença de **temperatura de densidade** entre uma parcela e o ambiente
quando a parcela sai do nível de origem (eixo x) e é levantada até o nível do eixo y.
* **Vermelho (> 0):** parcela mais leve que o ambiente — flutuabilidade positiva.
* **Azul (< 0):** parcela mais densa — flutuabilidade negativa (inibição).
* **Linha grossa (0 K):** separa as duas regiões; a parte de cima da região vermelha marca o nível de equilíbrio da parcela.

**2. Reversível × pseudoadiabático.**
* **Reversível:** toda a água condensada fica na parcela (peso do condensado e calor do líquido) → flutuabilidade menor.
* **Pseudoadiabático:** o condensado precipita assim que se forma → flutuabilidade maior.
A diferença entre os dois mostra o quanto o **carregamento de água** pode reduzir a energia disponível.

**3. Perguntas para a sua sondagem** (responda com os valores da tabela `cape_emanuel` e dos gráficos — não use números de outros casos):
* De quais níveis de origem as parcelas têm flutuabilidade positiva? Até que altura?
* Existe uma camada azul logo acima da origem (inibição)? Qual a sua espessura?
* Qual a CAPE reversível e pseudoadiabática da parcela de superfície e qual a máxima (e de que nível de origem)?
* Os resultados concordam com a SBCAPE/MUCAPE do MetPy e com os índices do Wyoming? Se não, por quê?

---
---
# As três sondagens: estável, neutra e instável

Cada aluno escolhe **três sondagens reais** que representem os três regimes. Edite a lista `CASOS` abaixo
(estação, data/hora, fonte FM35/BUFR e o regime que você espera). O exemplo usa Porto Alegre (SBPA, 83971) em dezembro de 1995.

> A classificação de cada caso deve ser **confirmada pelos números** (CAPE, CIN, LI…). Se o caso escolhido como "neutro"
> tiver CAPE alta, troque a data — não force a interpretação.

In [ ]:
CASOS = [
    dict(regime='ESTÁVEL',  station='83971', date=datetime(1995, 12, 12, 12), src='FM35'),
    dict(regime='NEUTRA',   station='83971', date=datetime(1995, 12, 23, 12), src='FM35'),   # confirmar pelos números
    dict(regime='INSTÁVEL', station='83971', date=datetime(1995, 12, 24, 12), src='FM35'),
]

def indices_basicos(d):
    p = d['pressure'].values * units.hPa; T = d['temperature'].values * units.degC; Td = d['dewpoint'].values * units.degC
    z = d['height'].values * units.m; h = z - z[0]
    u, v = mpcalc.wind_components(d['speed'].values * units.knots, d['direction'].values * units.degrees)
    w = (d['speed'].notna() & d['direction'].notna()).values
    prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    sb = mpcalc.surface_based_cape_cin(p, T, Td); mu = mpcalc.most_unstable_cape_cin(p, T, Td)
    ml = mpcalc.mixed_layer_cape_cin(p, T, Td, depth=100 * units.hPa)
    rm, lm, _ = mpcalc.bunkers_storm_motion(p[w], u[w], v[w], h[w])
    s6 = mpcalc.bulk_shear(p[w], u[w], v[w], height=h[w], depth=6 * units.km)
    out = {
        'SBCAPE (J/kg)': sb[0].m, 'SBCIN (J/kg)': sb[1].m, 'MLCAPE (J/kg)': ml[0].m, 'MUCAPE (J/kg)': mu[0].m,
        'LI (K)': mpcalc.lifted_index(p, T, prof)[0].m, 'K (°C)': mpcalc.k_index(p, T, Td).m,
        'PW (mm)': mpcalc.precipitable_water(p, Td).to('mm').m,
        'LCL (hPa)': mpcalc.lcl(p[0], T[0], Td[0])[0].m,
        'Shear 0-6 km (m/s)': mpcalc.wind_speed(*s6).to('m/s').m,
        'SRH 0-3 km LM (m²/s²)': mpcalc.storm_relative_helicity(h[w], u[w], v[w], depth=3*units.km,
                                                               storm_u=lm[0], storm_v=lm[1])[2].m,
    }
    return out, dict(p=p, T=T, Td=Td, u=u, v=v, w=w, prof=prof)

for c in CASOS:
    d, c['fonte'], c['indices_wyoming'] = busca_wyoming(c['station'], c['date'], c['src'])
    c['df'] = limpa_sondagem(d)
    c['indices'], c['perfil'] = indices_basicos(c['df'])
    c['rotulo'] = f"{c['regime']} — {c['station']} {c['date']:%d/%m/%Y %HZ}"
    print(f"{c['rotulo']}: {len(c['df'])} níveis — {c['fonte']}")

tabela3 = pd.DataFrame({c['rotulo']: c['indices'] for c in CASOS}).round(1)
tabela3

### Skew-T das três sondagens

In [ ]:
fig = plt.figure(figsize=(21, 8.5))
for k, c in enumerate(CASOS):
    P = c['perfil']
    skew = SkewT(fig, rotation=45, subplot=(1, 3, k + 1))
    skew.plot(P['p'], P['T'], 'r', lw=2, label='T')
    skew.plot(P['p'], P['Td'], 'g', lw=2, label='Td')
    skew.plot(P['p'], P['prof'], 'k--', lw=1.2, label='parcela de superfície')
    skew.shade_cape(P['p'], P['T'], P['prof'], color='lightcoral', alpha=0.4)
    skew.shade_cin(P['p'], P['T'], P['prof'], color='skyblue', alpha=0.4)
    pw_, uw_, vw_ = P['p'][P['w']], P['u'][P['w']], P['v'][P['w']]
    passo = max(1, len(pw_) // 35)
    skew.plot_barbs(pw_[::passo], uw_[::passo], vw_[::passo])
    skew.plot_dry_adiabats(lw=0.4); skew.plot_moist_adiabats(lw=0.4); skew.plot_mixing_lines(lw=0.4)
    skew.ax.set_ylim(1050, 100); skew.ax.set_xlim(-40, 45)
    I = c['indices']
    skew.ax.set_title(f"{c['rotulo']}\nSBCAPE {I['SBCAPE (J/kg)']:.0f} | MUCAPE {I['MUCAPE (J/kg)']:.0f} J/kg | "
                      f"LI {I['LI (K)']:.1f} K", fontsize=11)
    skew.ax.set_xlabel('Temperatura (°C)'); skew.ax.set_ylabel('Pressão (hPa)' if k == 0 else '')
    if k == 0: skew.ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('skewt_3_sondagens.png', dpi=150, bbox_inches='tight')
plt.show()

### θ, θe e θes das três sondagens (mesmos eixos)

In [ ]:
P_TOPO = 300
fig, axs = plt.subplots(1, 3, figsize=(17, 7), sharey=True)
perfis = []
for c in CASOS:
    P = c['perfil']
    perfis.append((mpcalc.potential_temperature(P['p'], P['T']).m,
                   mpcalc.equivalent_potential_temperature(P['p'], P['T'], P['Td']).m,
                   mpcalc.saturation_equivalent_potential_temperature(P['p'], P['T']).m, P['p'].m))
vals = np.concatenate([np.concatenate([a[pp >= P_TOPO] for a in (th, the, thes)]) for th, the, thes, pp in perfis])
x0, x1 = np.floor(vals.min() / 5) * 5 - 2, np.ceil(vals.max() / 5) * 5 + 2
for ax, c, (th, the, thes, pp) in zip(axs, CASOS, perfis):
    ax.plot(th, pp, color='tab:blue', lw=2, label='θ')
    ax.plot(the, pp, color='tab:red', lw=2, label='θe')
    ax.plot(thes, pp, color='tab:green', lw=2, ls='--', label='θes')
    ax.set_yscale('log'); ax.set_ylim(1050, P_TOPO); ax.set_xlim(x0, x1); ax.grid(alpha=0.3)
    tk = [t for t in (1000, 925, 850, 700, 600, 500, 400, 300) if t >= P_TOPO]
    ax.set_yticks(tk); ax.set_yticklabels(map(str, tk)); ax.minorticks_off()
    ax.set_xlabel('Temperatura potencial (K)'); ax.set_title(c['rotulo'], fontsize=11); ax.legend(loc='upper left')
axs[0].set_ylabel('Pressão (hPa)')
plt.savefig('theta_3_sondagens.png', dpi=150, bbox_inches='tight')
plt.show()

### Perturbação de temperatura de densidade de Emanuel — três sondagens

O `wyoming.exe` compilado acima roda para cada sondagem, numa pasta própria. As três linhas de figuras usam a
**mesma escala de cores** para que os regimes possam ser comparados diretamente.

In [ ]:
EM = {}
for c in CASOS:
    pasta = f"emanuel_{c['date']:%Y%m%d_%H}"
    os.makedirs(pasta, exist_ok=True)
    escreve_sounding_txt(c['df'], c['station'], c['date'], os.path.join(pasta, 'sounding.txt'))
    roda_wyoming(pasta)
    E = dict(p=np.loadtxt(f'{pasta}/p.out'), porig=np.loadtxt(f'{pasta}/porig.out'),
             rev=np.loadtxt(f'{pasta}/tdifrev.out'), pse=np.loadtxt(f'{pasta}/tdifpseudo.out'),
             cape=le_cape_out(f'{pasta}/cape.out'))
    E['X'], E['Y'] = np.meshgrid(E['porig'], E['p'])
    EM[c['rotulo']] = E

amp = 6.0
for E in EM.values():
    vis = (E['Y'] < E['X']) & (E['Y'] >= 100)
    amp = max(amp, 2 * np.ceil(max(np.nanmax(E['rev'][vis]), np.nanmax(E['pse'][vis])) / 2))

fig, axs = plt.subplots(2, 3, figsize=(20, 14), sharey=True, gridspec_kw=dict(hspace=0.35))
for j, (rot, E) in enumerate(EM.items()):
    for i, (campo, nome) in enumerate(((E['rev'], 'reversível'), (E['pse'], 'pseudoadiabática'))):
        cf = plota_emanuel(axs[i, j], E['X'], E['Y'], campo, amp)
        k = E['cape']['CAPE_pse' if i else 'CAPE_rev'].idxmax()
        col = 'CAPE_pse' if i else 'CAPE_rev'
        axs[i, j].set_title(f"{rot}\nascensão {nome} — CAPE máx {E['cape'].loc[k, col]:.0f} J/kg "
                            f"(origem {E['cape'].loc[k, 'p_origem']:.0f} hPa)", fontsize=10)
        if j: axs[i, j].set_ylabel('')
fig.colorbar(cf, ax=axs, orientation='horizontal', fraction=0.03, pad=0.07, ticks=np.arange(-amp, amp + 1, 2),
             label='diferença de temperatura de densidade parcela − ambiente (K)' +
                   ('' if CORTE_EMANUEL else ' — sem o corte de −4 K do original'))
plt.savefig('emanuel_3_sondagens.png', dpi=150, bbox_inches='tight')
plt.show()

resumo = pd.DataFrame({rot: {'CAPE rev. superfície (J/kg)': E['cape'].iloc[0]['CAPE_rev'],
                             'CAPE pseudo. superfície (J/kg)': E['cape'].iloc[0]['CAPE_pse'],
                             'CAPE rev. máx (J/kg)': E['cape']['CAPE_rev'].max(),
                             'CAPE pseudo. máx (J/kg)': E['cape']['CAPE_pse'].max(),
                             'origem da CAPE pseudo. máx (hPa)': E['cape'].loc[E['cape']['CAPE_pse'].idxmax(), 'p_origem'],
                             'DCAPE máx Emanuel (J/kg)': E['cape']['DCAPE'].max()}
                       for rot, E in EM.items()}).round(1)
resumo

### Para a apresentação
* Use **somente** números das tabelas `tabela3` e `resumo` (e das células da sondagem individual) — e confira com os índices do Wyoming.
* Compare os três regimes: onde está a flutuabilidade positiva, quão espessa é a inibição, qual a diferença reversível × pseudoadiabática.
* Afirmações sobre o **evento** (chuva, granizo, danos) precisam de fonte externa citada.